1. Loading both files


In [8]:
import pandas as pd

df_gateway = pd.read_csv('/gateway.csv')
df_ledger = pd.read_csv('/ledger.csv')
display(df_gateway)
display(df_ledger)




,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI
5,R007,2026-03-04,M005,3300.0,failed,NetBanking
6,R008,2026-03-04,M001,600.0,success,Card
7,R009,2026-03-05,M002,4100.0,success,Card
8,R011,2026-03-05,M003,1800.0,success,Card


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card
5,R006,2026-03-03,M002,950.0,success,UPI
6,R007,2026-03-04,M005,3300.0,failed,NetBanking
7,R008,2026-03-04,M001,640.0,success,Card
8,R009,2026-03-05,M002,4100.0,success,Card
9,R010,2026-03-05,M004,2500.0,success,Wallet


2. Check duplicates and nulls

In [9]:
# 1. Check for Null (Missing) Values
null_counts = df_gateway.isnull().sum()
print("\n[Null Value Report]")
if null_counts.sum() == 0:
  print("Success: No null values found.")
else:
  print(null_counts[null_counts > 0])


[Null Value Report]
Success: No null values found.


In [10]:
# 1. Check for Null (Missing) Values
null_counts = df_ledger.isnull().sum()
print("\n[Null Value Report]")
if null_counts.sum() == 0:
  print("Success: No null values found.")
else:
  print(null_counts[null_counts > 0])


[Null Value Report]
Success: No null values found.


In [11]:
# 2. Check for Duplicates (Entire Row)
duplicate_rows = df_gateway.duplicated().sum()
print(f"\n[Duplicate Row Report]\nTotal identical rows: {duplicate_rows}")


[Duplicate Row Report]
Total identical rows: 0


In [12]:
# 2. Check for Duplicates (Entire Row)
duplicate_rows = df_ledger.duplicated().sum()
print(f"\n[Duplicate Row Report]\nTotal identical rows: {duplicate_rows}")


[Duplicate Row Report]
Total identical rows: 0


In [13]:
# 3. Check for Unique Key Duplicates (e.g., transaction_id)
    # Note: Replace 'transaction_id' with the actual ID column name in your CSVs
id_col = 'transaction_id'
if id_col in df_gateway.columns:
  id_duplicates = df_gateway.duplicated(subset=[id_col]).sum()
  print(f"Duplicate {id_col} entries: {id_duplicates}")
else:
  print(f"Warning: Column '{id_col}' not found for specific ID check.")

print("-" * 30 + "\n")

Duplicate transaction_id entries: 0
------------------------------



In [14]:
# 3. Check for Unique Key Duplicates (e.g., transaction_id)
    # Note: Replace 'transaction_id' with the actual ID column name in your CSVs
id_col = 'transaction_id'
if id_col in df_ledger.columns:
  id_duplicates = df_ledger.duplicated(subset=[id_col]).sum()
  print(f"Duplicate {id_col} entries: {id_duplicates}")
else:
  print(f"Warning: Column '{id_col}' not found for specific ID check.")

print("-" * 30 + "\n")

Duplicate transaction_id entries: 0
------------------------------



3. Identify records missing in gateway

In [15]:
from google.colab import files

# Get all transactions IDs from both sources
ledger_ids = set(df_ledger['transaction_id'])
gateway_ids = set(df_gateway['transaction_id'])

print(ledger_ids)
print(f"Unique IDs in Internal Ledger : {len(ledger_ids)}")
print(gateway_ids)
print(f"Unique IDs in Gateway Export : {len(gateway_ids)}")

{'R007', 'R006', 'R010', 'R005', 'R001', 'R009', 'R004', 'R003', 'R002', 'R008'}
Unique IDs in Internal Ledger : 10
{'R007', 'R006', 'R005', 'R001', 'R009', 'R011', 'R003', 'R002', 'R008'}
Unique IDs in Gateway Export : 9


In [16]:
#Find IDs in ledger but not in gateway

missing_in_gateway = ledger_ids - gateway_ids
print(f"Transactions missing in Gateway : {len(missing_in_gateway)}")
print(f"Transaction IDs : {missing_in_gateway}")

# Convert the set of IDs to a DataFrame to use .to_csv()
import pandas as pd
missing_in_gateway_df_for_export = pd.DataFrame(list(missing_in_gateway), columns=['transaction_id'])
missing_in_gateway_df_for_export.to_csv('missing_in_gateway_ids.csv', index=False)

# Trigger the browser download
from google.colab import files
files.download('missing_in_gateway_ids.csv')

Transactions missing in Gateway : 2
Transaction IDs : {'R004', 'R010'}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

4. Identify records missing in ledger

In [17]:
#Find IDs in gateway but not in ledger

missing_in_ledger = gateway_ids - ledger_ids
print(f"Transactions missing in Ledger : {len(missing_in_ledger)}")
print(f"Transaction IDs : {missing_in_ledger}")

# Convert the set of IDs to a DataFrame to use .to_csv()
import pandas as pd
missing_in_ledger_df_for_export = pd.DataFrame(list(missing_in_ledger), columns=['transaction_id'])
missing_in_ledger_df_for_export.to_csv('missing_in_ledger_ids.csv', index=False)

# Trigger the browser download
from google.colab import files
files.download('missing_in_ledger_ids.csv')

Transactions missing in Ledger : 1
Transaction IDs : {'R011'}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

5. Identify amount mismatches

In [18]:
#Get full details of missing transactions
#we need to filter the ledger for these IDs

missing_in_gateway_list = list(missing_in_gateway)
missing_in_gateway_df = df_ledger[df_ledger['transaction_id'].isin(missing_in_gateway_list)].copy()

#Add a column to mark the issue type
missing_in_gateway_df['issue_type'] = 'MISSING IN GATEWAY'

#Calculate total risk
total_missing_value = missing_in_gateway_df['amount_usd'].sum()
print(f"\nTotal amount of missing transactions : ₹{total_missing_value:,}")
missing_in_gateway_df


Total amount of missing transactions : ₹4,600.0


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method,issue_type
3,R004,2026-03-02,M003,2100.0,success,Card,MISSING IN GATEWAY
9,R010,2026-03-05,M004,2500.0,success,Wallet,MISSING IN GATEWAY


In [19]:
#Get fulldetails
missing_in_ledger_list = list(missing_in_ledger)
missing_in_ledger_df = df_gateway[df_gateway['transaction_id'].isin(missing_in_ledger_list)].copy()

#Add Issue type
missing_in_ledger_df['issue_type'] = 'MISSING IN LEDGER'

#Calculate total risk
total_unrecorded_value = missing_in_ledger_df['amount_usd'].sum()
print(f"\nTotal amount of missing transactions : ₹{total_unrecorded_value:,}")
missing_in_ledger_df


Total amount of missing transactions : ₹1,800.0


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method,issue_type
8,R011,2026-03-05,M003,1800.0,success,Card,MISSING IN LEDGER


In [20]:
#Step 3: find amount mismatch

#Find common transactions(present in both)

common_ids = ledger_ids & gateway_ids
print(f"Transactions with mismatched amounts : {len(common_ids)}")
print(f"Transaction IDs : {common_ids}")
#Get data for common transactions from both sources
common_ids_list = list(common_ids)

ledger_common = df_ledger[df_ledger['transaction_id'].isin(common_ids)].copy()
gateway_common = df_gateway[df_gateway['transaction_id'].isin(common_ids)].copy()

print(f"Ledger Common : {len(ledger_common)}")
print(f"Gateway Common : {len(gateway_common)}")
#Merge on transaction_id to compare side by side

comparison = ledger_common.merge(
gateway_common[['transaction_id', 'amount_usd', 'status']],
    on='transaction_id',
    suffixes=('_ledger', '_gateway')
)
print(f"Merged comparison table : {len(comparison)}")
comparison

Transactions with mismatched amounts : 8
Transaction IDs : {'R007', 'R006', 'R005', 'R009', 'R003', 'R002', 'R001', 'R008'}
Ledger Common : 8
Gateway Common : 8
Merged comparison table : 8


,transaction_id,transaction_date,merchant_id,amount_usd_ledger,status_ledger,payment_method,amount_usd_gateway,status_gateway
0,R001,2026-03-01,M001,1200.0,success,UPI,1200.0,success
1,R002,2026-03-01,M002,850.0,success,Card,900.0,success
2,R003,2026-03-02,M001,500.0,success,Wallet,500.0,success
3,R005,2026-03-03,M004,7200.0,success,Card,7200.0,failed
4,R006,2026-03-03,M002,950.0,success,UPI,950.0,success
5,R007,2026-03-04,M005,3300.0,failed,NetBanking,3300.0,failed
6,R008,2026-03-04,M001,640.0,success,Card,600.0,success
7,R009,2026-03-05,M002,4100.0,success,Card,4100.0,success


In [21]:
# Calculate the difference in amounts
comparison['amount_difference'] = comparison['amount_usd_ledger'] - comparison['amount_usd_gateway']

# Filter for transactions where amounts do not match
amount_mismatch_df = comparison[comparison['amount_difference'] != 0].copy()

# Add issue type
amount_mismatch_df['issue_type'] = 'AMOUNT MISMATCH'

# Calculate total risk
total_mismatch_value = amount_mismatch_df['amount_difference'].abs().sum()

print(f"Number of amount mismatches: {len(amount_mismatch_df)}")
print(f"Total absolute amount discrepancy: ₹{total_mismatch_value:,}")
print("Details of amount mismatches:")
amount_mismatch_df[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'amount_difference', 'issue_type']]

Number of amount mismatches: 2
Total absolute amount discrepancy: ₹90.0
Details of amount mismatches:


,transaction_id,amount_usd_ledger,amount_usd_gateway,amount_difference,issue_type
1,R002,850.0,900.0,-50.0,AMOUNT MISMATCH
6,R008,640.0,600.0,40.0,AMOUNT MISMATCH


Status Mismatch

In [22]:
#Step 4 : Find the status mismatch
import pandas as pd

status_mismatches_series =  comparison['status_ledger']!=comparison['status_gateway']
status_mismatch_df = comparison[status_mismatches_series].copy()

print(f"Number of status mismatches : {status_mismatch_df.shape[0]}")
if status_mismatch_df.shape[0] > 0:
  print("Details of status mismatches:")
  display(status_mismatch_df[['transaction_id', 'status_ledger', 'status_gateway']])

# Save the status mismatch data to a CSV
status_mismatch_df.to_csv('status_mismatches.csv', index=False)

# Trigger the browser download
from google.colab import files
files.download('status_mismatches.csv')

Number of status mismatches : 1
Details of status mismatches:


,transaction_id,status_ledger,status_gateway
3,R005,success,failed


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
# Prepare amount mismatches for the report
amount_mismatch_report = amount_mismatch_df[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'amount_difference']].copy()
amount_mismatch_report['issue_type'] = 'AMOUNT_MISMATCH'
amount_mismatch_report.columns = ['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'difference', 'issue_type']

print(f"Amount mismatch records: {len(amount_mismatch_report)}")
# amount_mismatch_report # Commented out to prevent duplicate display in final execution

# Prepare missing in gateway for the report
missing_gateway_report = missing_in_gateway_df[['transaction_id', 'amount_usd']].copy()
missing_gateway_report['gateway_amount'] = 0 # Gateway amount is 0 for missing in gateway
missing_gateway_report['difference'] = missing_gateway_report['amount_usd'] # Difference is ledger_amount - 0
missing_gateway_report['issue_type'] = 'MISSING_IN_GATEWAY'
# Rename columns to match the standard report format
missing_gateway_report.columns = ['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'difference', 'issue_type']

print(f"Missing in gateway records: {len(missing_gateway_report)}")
# missing_gateway_report # Commented out to prevent duplicate display in final execution

# Prepare missing in ledger for the report
missing_ledger_report = missing_in_ledger_df[['transaction_id', 'amount_usd']].copy()
# 'amount_usd' here is the gateway amount for these transactions, so rename it
missing_ledger_report = missing_ledger_report.rename(columns={'amount_usd': 'amount_usd_gateway'})
missing_ledger_report['amount_usd_ledger'] = 0 # Ledger amount is 0 for missing in ledger
missing_ledger_report['difference'] = -missing_ledger_report['amount_usd_gateway']  # Negative because it's only in gateway
missing_ledger_report['issue_type'] = 'MISSING_IN_LEDGER'

# Reorder columns to match the standard report format
missing_ledger_report = missing_ledger_report[[
    'transaction_id',
    'amount_usd_ledger',
    'amount_usd_gateway',
    'difference',
    'issue_type'
]]

print(f"Missing in ledger records: {len(missing_ledger_report)}")
# missing_ledger_report # Commented out to prevent duplicate display in final execution

# Prepare status mismatches for the report
# Use existing amount columns if available, and amount_difference
status_mismatch_report = status_mismatch_df[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'amount_difference']].copy()
status_mismatch_report['issue_type'] = 'STATUS_MISMATCH'
status_mismatch_report.columns = ['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'difference', 'issue_type']

print(f"Status mismatch records: {len(status_mismatch_report)}")

# Combine all issues into one report
reconciliation_report = pd.concat([
    missing_gateway_report,
    missing_ledger_report,
    amount_mismatch_report,
    status_mismatch_report # Add status mismatch report
], ignore_index=True)

print(f"\n=== RECONCILIATION REPORT ===")
print(f"Total issues found: {len(reconciliation_report)}")

# Count by issue type
print(f"\nBy issue type:")
print(reconciliation_report['issue_type'].value_counts())

# Sort by absolute difference (biggest issues first)
reconciliation_report['abs_difference'] = reconciliation_report['difference'].abs()
reconciliation_report = reconciliation_report.sort_values('abs_difference', ascending=False)
reconciliation_report = reconciliation_report.drop('abs_difference', axis=1)

# Reset index for clean display
reconciliation_report = reconciliation_report.reset_index(drop=True)

print("\nFull report (sorted by impact):")
display(reconciliation_report)

Amount mismatch records: 2
Missing in gateway records: 2
Missing in ledger records: 1
Status mismatch records: 1

=== RECONCILIATION REPORT ===
Total issues found: 6

By issue type:
issue_type
MISSING_IN_GATEWAY    2
AMOUNT_MISMATCH       2
MISSING_IN_LEDGER     1
STATUS_MISMATCH       1
Name: count, dtype: int64

Full report (sorted by impact):


,transaction_id,amount_usd_ledger,amount_usd_gateway,difference,issue_type
0,R010,2500.0,0.0,2500.0,MISSING_IN_GATEWAY
1,R004,2100.0,0.0,2100.0,MISSING_IN_GATEWAY
2,R011,0.0,1800.0,-1800.0,MISSING_IN_LEDGER
3,R002,850.0,900.0,-50.0,AMOUNT_MISMATCH
4,R008,640.0,600.0,40.0,AMOUNT_MISMATCH
5,R005,7200.0,7200.0,0.0,STATUS_MISMATCH


In [24]:
# Save the reconciliation report to a CSV file
reconciliation_report.to_csv('reconciliation_report.csv', index=False)

# Trigger the download of the reconciliation report
from google.colab import files
files.download('reconciliation_report.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
# Calculate summary statistics
total_ledger = df_ledger['amount_usd'].sum()
total_gateway_success = df_gateway[df_gateway['status'] == 'success']['amount_usd'].sum()

# Build summary dictionary
summary = {}
summary['Total Transactions in Ledger'] = len(df_ledger)
snapshot_total_gateway = len(df_gateway) # store this in a variable before it gets overridden by the next line.
summary['Total Transactions in Gateway'] = snapshot_total_gateway
summary['Matched Transactions'] = len(common_ids) - len(amount_mismatch_df)
summary['Total Issues Found'] = len(reconciliation_report)
summary['Missing in Gateway (Count)'] = len(missing_in_gateway)
summary['Missing in Gateway (Value)'] = missing_in_gateway_df['amount_usd'].sum()
summary['Missing in Ledger (Count)'] = len(missing_in_ledger)
summary['Missing in Ledger (Value)'] = missing_in_ledger_df['amount_usd'].sum()
summary['Amount Mismatches (Count)'] = len(amount_mismatch_df)

if len(amount_mismatch_df) > 0:
    summary['Amount Mismatches (Net Diff)'] = amount_mismatch_df['amount_difference'].sum()
elif len(amount_mismatch_df) == 0: # Handle the case where amount_mismatch_df might be empty
    summary['Amount Mismatches (Net Diff)'] = 0

summary['Ledger Total'] = total_ledger
summary['Gateway Total (SUCCESS only)'] = total_gateway_success

# Display as DataFrame
summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
summary_df



,Metric,Value
0,Total Transactions in Ledger,10.0
1,Total Transactions in Gateway,9.0
2,Matched Transactions,6.0
3,Total Issues Found,6.0
4,Missing in Gateway (Count),2.0
5,Missing in Gateway (Value),4600.0
6,Missing in Ledger (Count),1.0
7,Missing in Ledger (Value),1800.0
8,Amount Mismatches (Count),2.0
9,Amount Mismatches (Net Diff),-10.0


In [29]:
import json

# 1. Calculate the values
summary_data = {
    "total_ledger_rows": len(df_ledger),
    "total_gateway_rows": len(df_gateway),
    "missing_in_gateway_count": len(reconciliation_report[reconciliation_report['issue_type'] == 'MISSING_IN_GATEWAY']),
    "missing_in_ledger_count": len(reconciliation_report[reconciliation_report['issue_type'] == 'MISSING_IN_LEDGER']),
    "amount_mismatch_count": len(reconciliation_report[reconciliation_report['issue_type'] == 'AMOUNT_MISMATCH']),
    "status_mismatch_count": len(reconciliation_report[reconciliation_report['issue_type'] == 'STATUS_MISMATCH']),
    "reconciliation_issue_count": len(reconciliation_report),
    "ledger_total_amount": float(df_ledger['amount_usd'].sum()),
    "gateway_total_amount": float(df_gateway[df_gateway['status'] == 'success']['amount_usd'].sum()),
    "amount_at_risk": float(reconciliation_report['difference'].abs().sum())
}

# 2. Save to JSON file
with open('summary_report.json', 'w') as f:
    json.dump(summary_data, f, indent=2)

# 3. Print the result to verify
print(json.dumps(summary_data, indent=2))

# 4. Trigger download
from google.colab import files
files.download('summary_report.json')

{
  "total_ledger_rows": 10,
  "total_gateway_rows": 9,
  "missing_in_gateway_count": 2,
  "missing_in_ledger_count": 1,
  "amount_mismatch_count": 2,
  "status_mismatch_count": 1,
  "reconciliation_issue_count": 6,
  "ledger_total_amount": 23340.0,
  "gateway_total_amount": 10050.0,
  "amount_at_risk": 6490.0
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
import pandas as pd

# Read the JSON file into a DataFrame
df_json = pd.read_json('api_response_sample.json')

# Display the first few rows
df_json.head()

FileNotFoundError: File api_response_sample.json does not exist

In [ ]:
import pandas as pd
import json

# 1. Load the raw JSON data
with open('api_response_sample.json', 'r') as f:
    data = json.load(f)

# 2. Flatten the 'batches' nested structure
# This will turn keys like 'merchant_id' inside 'batches' into columns
df_flat = pd.json_normalize(data, record_path=['batches'], meta=['generated_at', 'source'])

# 3. View the clean tabular result
display(df_flat.head())

In [ ]:
import pandas as pd
import json

# 1. Load the raw data
with open('api_response_sample.json', 'r') as f:
    data = json.load(f)

# 2. Flatten at the 'settlements' level
# We go inside 'batches' then inside 'settlements'
df_deep = pd.json_normalize(
    data,
    record_path=['batches', 'settlements'],
    meta=[
        'generated_at',
        ['batches', 'batch_id'],
        ['batches', 'merchant', 'merchant_id'],
        ['batches', 'merchant', 'merchant_name']
    ]
)

# 3. Clean the column names (dots to underscores)
df_deep.columns = [col.replace('.', '_').lower() for col in df_deep.columns]

# 4. Final touch: rename the meta columns for clarity
df_deep = df_deep.rename(columns={
    'batches_batch_id': 'batch_id',
    'batches_merchant_merchant_id': 'merchant_id',
    'batches_merchant_merchant_name': 'merchant_name'
})

display(df_deep.head())

In [ ]:
# 1. Convert the date columns to datetime objects
df_deep['processed_at'] = pd.to_datetime(df_deep['processed_at'])
df_deep['generated_at'] = pd.to_datetime(df_deep['generated_at'])

# 2. Verify the data types
print(df_deep[['processed_at', 'generated_at']].dtypes)

# 3. View the result
display(df_deep.head())

In [ ]:
# Save to the current Colab directory
df_deep.to_csv('normalized_settlements.csv', index=False)

# Optional: Trigger an automatic browser download to your computer
from google.colab import files
files.download('normalized_settlements.csv')
